In [1]:
import os 
from dotenv import load_dotenv 
load_dotenv()

True

In [2]:
if os.environ["OPENAI_API_KEY"]:
    print("API key is set")

API key is set


In [3]:
from langchain_openai import ChatOpenAI

In [4]:
llm= ChatOpenAI(model='gpt-5-nano',temperature=0) # to send requuest to model , temp=0 no creative answers

**RAG IMPLEMENTATION WITH OWNN PDF**

## STEP 1 Extracting text from PDF

In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path= "./fabric-admin.pdf"

loader=PyPDFLoader(pdf_path)
docs=loader.load()

docs

/var/folders/km/n5txgrn17132qnzvdz2swxbc0000gn/T/ipykernel_6833/3132491757.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'Microsoft Learn PDF 1.0.26144.02', 'creator': 'Microsoft Learn', 'creationdate': '2026-07-13T16:01:57+00:00', 'title': 'fabric admin | Microsoft Learn', 'moddate': '2026-07-13T16:01:57+00:00', 'source': './fabric-admin.pdf', 'total_pages': 316, 'page': 0, 'page_label': '1'}, page_content='Fabric administration documentation\nLearn about the Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is administration in Fabric?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification\nConfigure notifications\nSet up metadata scanning'),
 Document(metadata={'producer': 'Microsoft Learn PDF 1.0.26144.02', 'creator': 'Microsoft Learn', 'creationdate': '2026-07

## Step 1.1 Creating own meta data for PDF CHUNKS

In [6]:
for i in docs:
    i.metadata= {"source":"fabric-admin.pdf",
                 "developer":'microsoft'}

In [7]:
docs

[Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'microsoft'}, page_content='Fabric administration documentation\nLearn about the Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is administration in Fabric?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification\nConfigure notifications\nSet up metadata scanning'),
 Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'microsoft'}, page_content='Enable content certification\nEnable service principal authentication\nConfigure Multi-Geo support\nMonitoring and management\nｅOVERVIEW\nWhat is the admin monitoring workspace?\nｐCONCEPT\nFeature usage and adoption report\nｃHOW-TO GUIDE\nUse the Monitoring hub\n

## Step 2 Chunking
splitting documents to chunks

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document  

splitter= RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100
)

chunks= splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'microsoft'}, page_content='Fabric administration documentation\nLearn about the Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is administration in Fabric?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification\nConfigure notifications\nSet up metadata scanning'),
 Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'microsoft'}, page_content='Enable content certification\nEnable service principal authentication\nConfigure Multi-Geo support\nMonitoring and management\nｅOVERVIEW\nWhat is the admin monitoring workspace?\nｐCONCEPT\nFeature usage and adoption report\nｃHOW-TO GUIDE\nUse the Monitoring hub\n

In [10]:
chunks[0].metadata

{'source': 'fabric-admin.pdf', 'developer': 'microsoft'}

In [11]:
len(chunks)

574

## Step3 Creating embedding for the chunks

In [15]:
from langchain_openai import OpenAIEmbeddings

embedding_model= OpenAIEmbeddings(model='text-embedding-3-small') # by default model



## Step 4 Storage
Storing embedding in the existing local vecor store
1) We will add documents to existing vecorstore

CREATE AND STORE EMBEDDING IN LOCAL VECTOR STORE

In [16]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(
    persist_directory='./Vector/',
    embedding_function=embedding_model
)


## Step 4.1 Now we will add more documents to existing vector store

In [17]:
vectorstore.add_documents(chunks)

['f4267c1a-2628-4fc8-b86c-5075111744b5',
 '89c01c4e-6208-46db-8f60-578611235450',
 '9ceab972-fd4f-4177-94d0-60e26e164f07',
 '276131db-67b6-408f-878c-e8492415fa89',
 'f46ba68e-4a18-486c-a503-dd0e3c059ca7',
 '7502f86c-c496-4aae-853a-a251b12b4cd5',
 'efd4d143-d553-4161-bcb3-39ec5bc71867',
 '9eaa7560-b043-4676-8f4b-c774a5995e24',
 '34121e8e-062d-4f3d-b9d0-0043c1aa1f19',
 '63cc1615-437b-4c1d-9bd9-4e08f48f51e4',
 'f44cccb1-4354-46d0-a1e2-02fc769d5b41',
 '9cb07388-959f-4818-8639-9a6840ce2382',
 'f8397d71-e19b-452f-8a92-32a31bac36b3',
 '00536781-5504-4fca-9468-15f85b699fd7',
 '27fdc8e2-9880-4eca-a16b-4cf02e59c065',
 '9bccdcb7-dccf-4f89-a248-7e355923770b',
 '702d8bb9-1512-42fa-80d4-5c45296f5f7f',
 'e3fe6259-9796-4112-bebe-f3710f10e2c4',
 'b6b05424-4590-4d3d-bc66-08e8c8d09dbd',
 '06b69865-a278-4e9f-9915-14f649e73bb4',
 '98af8ab6-a7ce-4dba-a893-67524e9e1957',
 '23a41612-db79-415c-b615-2a667af96b4f',
 '41899a94-821b-441b-9dd2-b8e4d12f7c5c',
 '9c26e58d-32f7-4483-9fc7-29334fb75a1e',
 '9bb0801d-086b-

We have chunks provided by user+ embedding model (open AI model)+ metadata ==

Both these inputs are provided to langchain

Langchain will convert these inputs to 5 vector above 

and these 5 vetcor will save to Chroma DB automatically ]

it has run below loop tp save these vectors one by one along with metadeta

vector=[]
for doc in chunks :
vector=embedding_model.embed_documents([doc.page_content])
vector.append(vector)


Chroma DB will store vector+metadata   (Metadata+text)
like below
Document(Metadata={},page_content="_")

meaning storing vector+mapped_chunk


## Step5 Similarity search/ Semantic search

In [18]:
vectorstore.similarity_search('What is micrsoft fabric admin',k=3)

[Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'microsoft'}, page_content='What is Microsoft Fabric administration?\nFabric administration is the set of tasks and tools you use to configure, secure, and govern the\nFabric software as a service (SaaS) platform across your organization. As an admin, you control\ntenant-wide settings, manage feature access to meet company policies and regulations, and\ndelegate responsibilities so no single team becomes a bottleneck.\nThere are generally three categories of tasks that admins focus on to ensure the platform is\nconfigured correctly and compliant with organizational policies:\nAdministration—This Fabric administration documentation covers how to manage the\nFabric platform, configure tenant and workspace features, and monitor usage and activity.\nSecurity—See the Security documentation to learn how to help safeguard data with identity,\naccess, encryption, and network protection settings.\nGovernance—See the Governance docum

I have Chroma DB . I am user and do similarity search as above code

Chorma DB will return 3 vector 

my question will first converted to verctor store using model an this vector will do the similary serach in Chorma DB and then it will return top 3 vectors and these top 3 vectors will return mapped chunks and then it will drive the result 

** Step6 Talk to LLM

In [19]:
context=vectorstore.similarity_search('Why do we need one lake?',k=3)

In [20]:
llm.invoke(f'Why do we need one lake?, you can answer using folliwing context: {context}')

AIMessage(content='OneLake exists to be the central data lake inside Microsoft Fabric, so there’s a single, governed place to store and access data for all Fabric apps. Based on the provided material, here’s why we need it:\n\n- Centralized access for Fabric apps: Data stored in OneLake can be used by apps inside the Fabric environment (e.g., Spark, Data Engineering, Data Warehouse) via ADLS APIs, OneLake File Explorer, and Databricks. This reduces data silos and makes data readily usable across tools.\n\n- Interoperability with familiar tools: It works with standard interfaces and tools, allowing existing workflows to read and work with data without moving it elsewhere.\n\n- Secure, time-limited access: Applications access data through short-lived SAS tokens derived from a Fabric user’s Entra identity. These tokens are scoped for least-privilege access and cannot exceed a one-hour lifetime, helping keep data access tightly controlled.\n\n- Configurable governance and scale: OneLake se

In [21]:
response=llm.invoke(f'Why do we need one lake?,, you can answer using folliwing context: {context}')
response.content

'We need OneLake because it acts as the central, unified data store for the Microsoft Fabric ecosystem. Here’s what that buys us, based on the provided context:\n\n- Central access for all Fabric apps: Data stored in OneLake can be accessed by tools and workloads inside Fabric (e.g., Spark, Data Engineering, Data Warehouse, and Databricks) via ADLS APIs and the OneLake File Explorer. This avoids data silos and makes sharing data across services simpler.\n\n- Consistent interfaces: The same OneLake access points (ADLS APIs and File Explorer) are used across different tools, making it easier to work with data no matter which Fabric component you’re using.\n\n- Secure, controlled access: Access is granted through short-lived user-delegated SAS tokens based on a Fabric user’s Entra identity. These tokens can be restricted to least-privileged permissions and expire after at most one hour, reducing security risk if a token is exposed.\n\nIn short, OneLake provides a single, secure, and inter

## Step 7 Resuse the vector DB

In [22]:
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore_persist = Chroma(
    persist_directory="./Vector",
    embedding_function=embedding
)

In [23]:
vectorstore_persist.similarity_search('Why do we need one lake, k=3)')

[Document(metadata={'developer': 'microsoft', 'source': 'fabric-admin.pdf'}, page_content="(ADLS) APIs, OneLake File Explorer, and Databricks. Users can already access data\nstored in OneLake with apps internal to the Fabric environment, such as Spark,\nData Engineering, and Data Warehouse. Learn More\nUse short-lived user-\ndelegated SAS tokens\nOneLake SAS tokens enable applications to access data in OneLake through short-\nlived SAS tokens, based on a Microsoft Fabric user's Entra identity. These token's\npermissions can be further limited to provide least privileged access and cannot\nexceed a lifetime of one hour. Learn More\nDatamart settings\nﾉ Expand table\nSemantic model settings\nﾉ Expand table\nScale-out settings\nﾉ Expand table\nOneLake settings\nﾉ Expand table"),
 Document(metadata={'developer': 'microsoft', 'source': 'fabric-admin.pdf'}, page_content="(ADLS) APIs, OneLake File Explorer, and Databricks. Users can already access data\nstored in OneLake with apps internal to